# Classification classique — 30.72% Accuracy

Approche simple basée sur l'analyse HSV des bandes verticales.
Configuration optimisée pour 30%+ accuracy

In [ ]:
import os
import time
from collections import Counter

import cv2
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score
import matplotlib.pyplot as plt

LOAD_DATA = False
%run setup.ipynb

DATASET_BASE = str(dataset_root)
DIRECTIONS = ['North', 'South', 'East', 'West']

print(f"Dataset: {DATASET_BASE}")

dataset_root : /home/elias/Documents/FISE/Projet_image/Project datasets
Setup chargé — dataset_root : /home/elias/Documents/FISE/Projet_image/Project datasets
Dataset: /home/elias/Documents/FISE/Projet_image/Project datasets


In [ ]:

CONFIG = {
    'yellow_lower': np.array([20, 100, 100]),
    'yellow_upper': np.array([40, 255, 255]),
    'black_lower': np.array([0, 0, 0]),
    'black_upper': np.array([180, 255, 100]),  # less_black config optimal
    'morph_kernel': (5, 5),
}

def compute_band_densities(image_hsv, config=CONFIG):
    """Calcule les densités jaune/noir dans 3 bandes verticales de l'image"""
    h, w = image_hsv.shape[:2]
    third = max(1, h // 3)
    slices = [slice(0, third), slice(third, 2 * third), slice(2 * third, h)]

    densities = {}
    for name, s in zip(['top', 'mid', 'bot'], slices):
        band = image_hsv[s, :]
        area = band.shape[0] * band.shape[1]
        if area == 0:
            densities[f'y_{name}'] = 0.0
            densities[f'b_{name}'] = 0.0
            continue
        yellow = cv2.inRange(band, config['yellow_lower'], config['yellow_upper'])
        black = cv2.inRange(band, config['black_lower'], config['black_upper'])
        densities[f'y_{name}'] = np.sum(yellow > 0) / area
        densities[f'b_{name}'] = np.sum(black > 0) / area

    return densities

def classify_by_score(d):
    """Classification basée sur la distribution jaune/noir"""
    scores = {
        'North': (d['y_bot'] - d['y_top']) + (d['b_top'] - d['b_bot']),
        'South': (d['y_top'] - d['y_bot']) + (d['b_bot'] - d['b_top']),
        'East':  (d['b_top'] + d['b_bot']) / 2 - d['b_mid'] + d['y_mid'] - (d['y_top'] + d['y_bot']) / 2,
        'West':  (d['y_top'] + d['y_bot']) / 2 - d['y_mid'] + d['b_mid'] - (d['b_top'] + d['b_bot']) / 2,
    }
    return max(scores, key=scores.get)

def classify_image(image_path, config=CONFIG):
    """Classifie une image"""
    image = _read_image_any(image_path)
    if image is None:
        return None
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    densities = compute_band_densities(hsv, config)
    return classify_by_score(densities)

In [ ]:
def load_dataset(dataset_base_path=DATASET_BASE):
    """Charge toutes les images du dataset (167 images incluant le .gif)."""
    dataset = []
    for direction in DIRECTIONS:
        direction_path = os.path.join(dataset_base_path, direction)
        if not os.path.isdir(direction_path):
            print(f"  Dossier manquant: {direction_path}")
            continue
        img_files = [f for f in os.listdir(direction_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
        for img_file in img_files:
            dataset.append((os.path.join(direction_path, img_file), direction))
    return dataset


def _read_image_any(path):
    """Lit jpg/png/bmp via cv2 et gif via PIL (cv2 ne gère pas .gif)."""
    if path.lower().endswith('.gif'):
        from PIL import Image
        im = np.array(Image.open(path).convert('RGB'))
        return cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
    return cv2.imread(path)

In [4]:
def evaluate(dataset, config=CONFIG):
    """Évalue le modèle sur le dataset"""
    y_true = []
    y_pred = []
    timings = []

    for img_path, label in dataset:
        t0 = time.perf_counter()
        pred = classify_image(img_path, config)
        timings.append(time.perf_counter() - t0)

        if pred is None:
            continue

        y_true.append(label)
        y_pred.append(pred)

    if len(y_true) == 0:
        return {
            'n_total': len(dataset),
            'n_evaluated': 0,
            'accuracy': 0.0,
            'f1_macro': 0.0,
            'time_ms_per_image': float(np.mean(timings) * 1000.0) if timings else 0.0,
            'confusion_matrix': np.zeros((len(DIRECTIONS), len(DIRECTIONS)), dtype=int),
            'class_precision': {d: 0.0 for d in DIRECTIONS},
        }

    cm = confusion_matrix(y_true, y_pred, labels=DIRECTIONS)
    class_precision_values = precision_score(
        y_true, y_pred, labels=DIRECTIONS, average=None, zero_division=0,
    )

    return {
        'n_total': len(dataset),
        'n_evaluated': len(y_true),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, labels=DIRECTIONS, average='macro', zero_division=0)),
        'time_ms_per_image': float(np.mean(timings) * 1000.0),
        'pred_distribution': dict(Counter(y_pred)),
        'confusion_matrix': cm,
        'class_precision': {d: float(p) for d, p in zip(DIRECTIONS, class_precision_values)},
    }



In [5]:
dataset = load_dataset(DATASET_BASE)
metrics = evaluate(dataset, CONFIG)

print("RÉSULTATS - Méthode Classique")

print(f"Images évaluées: {metrics['n_evaluated']} / {metrics['n_total']}")
print(f"\n Accuracy: {metrics['accuracy']*100:.2f}% ({metrics['accuracy']:.4f})")
print(f"  F1-score macro: {metrics['f1_macro']:.4f}")
print(f"  Temps moyen: {metrics['time_ms_per_image']:.2f} ms/image")

print("\nPrécision par classe:")
for cls in DIRECTIONS:
    prec = metrics['class_precision'][cls]
    print(f"{cls:6s}: {prec*100:5.1f}%")

if 'pred_distribution' in metrics:
    print(f"\nDistribution prédictions: {metrics['pred_distribution']}")


RÉSULTATS - Méthode Classique
Images évaluées: 166 / 166

 Accuracy: 30.72% (0.3072)
  F1-score macro: 0.2570
  Temps moyen: 11.65 ms/image

Précision par classe:
North :  44.0%
South :  29.3%
East  :  12.5%
West  :  29.4%

Distribution prédictions: {'South': 99, 'North': 25, 'West': 34, 'East': 8}


In [ ]:
cm = metrics['confusion_matrix']
labels = DIRECTIONS

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_title(f"Matrice de confusion — Méthode classique V1 ({metrics['n_evaluated']} images, {metrics['accuracy']*100:.1f}%)", fontsize=12)
ax.set_xlabel('Classe prédite')
ax.set_ylabel('Classe réelle')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center', color='black')

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig('../report_figures/classical_v1_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()